# NY PREDATOR v1 - Entrenamiento en Google Colab (GPU)
## Bot especialista US30 (Apertura NY) - Entrenamiento PPO con 200,000 timesteps
### Instrucciones:
1. Sube el archivo US30_NY_ONLY_M15.csv usando la celda de carga de archivos
2. Cambia el entorno a GPU (Entorno de ejecución → Cambiar tipo de entorno → GPU T4)
3. Ejecuta todas las celdas
4. Descarga el modelo ONNX generado al final

## 1. Instalación de Dependencias

In [ ]:
!pip install stable-baselines3[extra] gymnasium pandas shimmy onnx onnxruntime onnxscript

## 2. Carga del Dataset desde Google Drive

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
from pathlib import Path
import logging
from datetime import datetime

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("Sube el archivo US30_NY_ONLY_M15.csv:")
uploaded = files.upload()

csv_filename = list(uploaded.keys())[0]
print(f"Archivo cargado: {csv_filename}")

## 3. Carga y Verificación del Dataset

In [ ]:
df = pd.read_csv(csv_filename, index_col=0, parse_dates=True)

logger.info(f"Dataset cargado: {len(df)} filas")
logger.info(f"Columnas: {df.columns.tolist()}")
logger.info(f"Rango de fechas: {df.index.min()} a {df.index.max()}")

required_columns = ['open', 'high', 'low', 'close', 'tickvol', 'spread', 'bb_upper', 'bb_lower', 
                    'bb_middle', 'bb_width', 'rsi', 'atr', 'target', 'volume_ma']

missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    logger.error(f"Faltan columnas requeridas: {missing_columns}")
else:
    logger.info("Todas las columnas requeridas están presentes")

print(f"\nDataset cargado exitosamente: {len(df)} filas")
print(f"Columnas: {df.columns.tolist()}")

## 4. Definición del Entorno de Trading (RL)

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import torch
from stable_baselines3 import PPO

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de entrenamiento: {device}")
if device == "cuda":
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")

class NYPredatorEnv(gym.Env):
    def __init__(self, df):
        super(NYPredatorEnv, self).__init__()
        self.df = df.reset_index(drop=True)
        self.current_step = 0
        self.max_steps = len(self.df) - 1
        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(10,), dtype=np.float32
        )
        self.balance = 100000
        self.position = 0
        self.entry_price = 0
        self.sl = 0
        self.tp = 0
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.balance = 100000
        self.position = 0
        self.entry_price = 0
        self.sl = 0
        self.tp = 0
        return self._get_observation(), {}
    
    def _get_observation(self):
        if self.current_step >= len(self.df):
            return np.zeros(10, dtype=np.float32)
        row = self.df.iloc[self.current_step]
        
        # Protección contra división por cero
        bb_upper = row['bb_upper'] if row['bb_upper'] != 0 else 1.0
        close = row['close'] if row['close'] != 0 else 1.0
        volume_ma = row['volume_ma'] if row['volume_ma'] != 0 else 1.0
        bb_width = row['bb_width'] if row['bb_width'] != 0 else 1.0
        
        obs = np.array([
            row['close'] / bb_upper,
            row['bb_width'],
            row['rsi'] / 100,
            row['atr'] / close,
            row['tickvol'] / volume_ma,  # Usando tickvol en lugar de volume
            (row['close'] - row['open']) / close,
            (row['high'] - row['low']) / close,
            (row['close'] - row['bb_lower']) / bb_width,
            (row['bb_upper'] - row['close']) / bb_width,
            self.position
        ], dtype=np.float32)
        return obs
    
    def _detectar_modo_a(self, row):
        mecha_superior = row['high'] > row['bb_upper']
        cierre_dentro = row['close'] < row['bb_upper'] and row['close'] > row['bb_lower']
        volumen_rechazo = row['tickvol'] > row['volume_ma']  # Usando tickvol en lugar de volume
        return mecha_superior and cierre_dentro and volumen_rechazo
    
    def _detectar_modo_b(self, row):
        cierre_fuera = row['close'] > row['bb_upper']
        bbw_anterior = (row['bb_upper'] - row['bb_lower']) / row['bb_middle']
        bbw_expansion = row['bb_width'] > bbw_anterior
        volume_anterior = self.df.iloc[self.current_step - 1]['tickvol'] if self.current_step > 0 else row['tickvol']  # Usando tickvol
        volumen_creciente = row['tickvol'] > volume_anterior  # Usando tickvol
        return cierre_fuera and bbw_expansion and volumen_creciente
    
    def _calcular_sl_tp(self, row):
        atr = row['atr']
        close = row['close']
        sl = close - (atr * 2.0)
        tp = close + (atr * 3.0)
        return sl, tp
    
    def step(self, action):
        if self.current_step >= self.max_steps:
            return self._get_observation(), 0, True, False, {}
        row = self.df.iloc[self.current_step]
        reward = 0
        if action == 0:
            if self._detectar_modo_a(row):
                sl, tp = self._calcular_sl_tp(row)
                self.position = -1
                self.entry_price = row['close']
                self.sl = sl
                self.tp = tp
                reward = 10
            else:
                reward = -1
        elif action == 1:
            if self._detectar_modo_b(row):
                sl, tp = self._calcular_sl_tp(row)
                self.position = 1
                self.entry_price = row['close']
                self.sl = sl
                self.tp = tp
                reward = 10
            else:
                reward = -1
        elif action == 2:
            reward = 0
        if self.position != 0:
            pnl = 0
            if self.position == 1:
                pnl = (row['close'] - self.entry_price) / self.entry_price * 100
            elif self.position == -1:
                pnl = (self.entry_price - row['close']) / self.entry_price * 100
            if row['close'] <= self.sl or row['close'] >= self.tp:
                self.position = 0
                reward += pnl
            else:
                reward += pnl * 0.1
        self.current_step += 1
        done = self.current_step >= self.max_steps
        return self._get_observation(), reward, done, False, {}

print("Entorno de trading definido exitosamente")

## 5. Entrenamiento del Modelo PPO (200,000 timesteps)

In [ ]:
env = NYPredatorEnv(df)

model = PPO(
    "MlpPolicy",
    env,
    learning_rate=0.0003,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    verbose=1,
    device=device,
    tensorboard_log="./logs/tensorboard"
)

print(f"Iniciando entrenamiento en {device}...")
print(f"Total timesteps: 200,000")
print("Tiempo estimado: 5-10 minutos en GPU T4")

model.learn(total_timesteps=200000)

print("\nEntrenamiento completado exitosamente")

## 6. Guardado del Modelo Entrenado

In [ ]:
import zipfile
import os

model.save("ny_predator_v1_FINAL")
print("Modelo guardado como ny_predator_v1_FINAL.zip")

with zipfile.ZipFile("ny_predator_v1_FINAL.zip", 'w') as zipf:
    zipf.write("ny_predator_v1_FINAL.zip")

print("Archivo ZIP creado exitosamente")

## 7. Conversión Automática a ONNX

In [ ]:
## 7. Conversión Automática a ONNX con NeuralPredatorMT5
import torch
import onnx
import torch.nn as nn

class NeuralPredatorMT5(nn.Module):
    """
    Wrapper para extracción de la red neuronal pura de PPO.
    Usa torch.argmax para compatibilidad con MetaTrader 5.
    opset_version=12 garantiza compatibilidad con MQL5.
    """
    def __init__(self, policy):
        super(NeuralPredatorMT5, self).__init__()
        self.policy = policy
        
    def forward(self, x):
        # Obtener logits del policy
        features = self.policy.features_extractor(x)
        latent_pi = self.policy.mlp_extractor.forward_actor(features)
        logits = self.policy.action_net(latent_pi)
        
        # Usar torch.argmax para compatibilidad con MQL5
        actions = torch.argmax(logits, dim=-1)
        
        return actions

# Crear wrapper del modelo
neural_predator = NeuralPredatorMT5(model.policy)
neural_predator.eval()

# Crear input dummy para exportación
dummy_input = torch.randn(1, 10).to(device)

# Exportar modelo a ONNX usando torch.onnx.export
torch.onnx.export(
    neural_predator,  # El wrapper NeuralPredatorMT5
    dummy_input,      # Input dummy
    "ny_predator_v1.onnx",  # Nombre del archivo de salida
    export_params=True,     # Exportar parámetros del modelo
    opset_version=12,       # Versión de ONNX 12 (compatibilidad MQL5)
    do_constant_folding=True,  # Optimizar constantes
    input_names=['input'],   # Nombre del input
    output_names=['output'], # Nombre del output
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

print("Modelo exportado a ONNX usando NeuralPredatorMT5")
print("Archivo: ny_predator_v1.onnx")
print("opset_version=12 para compatibilidad con MetaTrader 5")

## 8. Descarga del Modelo ONNX

In [ ]:
from google.colab import files

files.download("ny_predator_v1.onnx")

print("\n=== ENTRENAMIENTO COMPLETADO ===")
print("Modelo ONNX descargado: ny_predator_v1.onnx")
print("Este archivo debe ser colocado en MQL5/Files/ para usar en MT5")